# End-to-end pipeline

Runs the project modules in order. Calculations live in `src/epo_project`; this notebook only calls them and shows key outputs.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
import pandas as pd
from epo_project import config, data, replication

## Steps 1–2: industry returns, momentum and risk inputs

Downloads the French files (reused if already present), writes `data/processed` and `data/interim`, and checks the completion criteria.

In [2]:
summary = data.prepare_returns()
{k: v for k, v in summary.items() if k != "industries"}

{'first_month': '1926-07-31',
 'last_common_month': '2026-08-31',
 'missing_industry_months': 2868,
 'rf_source': 'F-F_Research_Data_Factors (monthly RF, one-month T-bill)',
 'eligibility_rule': 'momentum_ready: >= 12 consecutive months ending at t; risk_ready: >= 60; benchmark_eligible: both'}

In [3]:
data.build_inputs()

{'formation_dates': 1143,
 'first': '1931-06-30',
 'last': '2026-08-31',
 'industries_min_max': (40, 49),
 'smallest_eigenvalue_q': 3.545648451161342e-05}

## Step 3: Equity 1 benchmark validation

Raw MVO, MVO with Q and scalar EPO on the proposal grid, each scaled to 10% predicted volatility; w chosen each January from the preceding 60 months of Sharpe ratios. Published Equity 1 values are shown for comparison.

In [4]:
bench = replication.run()
bench["table"].round(3)

,portfolio,sharpe,sharpe_published,months,difference
0,Raw MVO (Sigma_hat),0.322,0.19,863,0.132
1,MVO with Q (EPO w=0),0.591,0.56,863,0.031
2,EPO w=0.25,0.741,0.75,863,-0.009
3,EPO w=0.50,0.790,0.79,863,-0.000
4,EPO w=0.75,0.816,0.80,863,0.016
5,EPO w=0.90,0.827,0.79,863,0.037
6,EPO w=0.95,0.831,NaN,863,NaN
7,EPO w=0.99,0.831,0.73,863,0.101
8,OOS EPO,0.770,0.79,863,-0.020


In [5]:
bench["diagnostics"].round(3)

,mean_eigenvalue,realized_to_predicted_vol,ex_ante_sharpe_relative,realized_sharpe_signal_direction,risk_share_mvo_q,risk_share_oos_epo
eigenvalue_group,,,,,,
1,0.067,1.963,0.963,0.158,0.188,0.037
2,0.127,1.584,1.015,-0.001,0.207,0.048
3,0.254,1.242,1.016,0.338,0.199,0.073
4,0.518,0.969,1.013,0.575,0.204,0.166
5,4.348,0.889,0.992,0.592,0.201,0.676
